# ADNI

## INIT

In [2]:
from data_model.DataCleaner import DataCleaner
from dl_client import DatalakeClient

dataCleaner = DataCleaner(support_file_path='ADNI_variables_statistics.xlsx')
client = DatalakeClient()

## ADNI MERGE

In [3]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'ADNIMERGE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

dataset = zip_files[list(zip_files.keys())[0]]

NameError: name 'client' is not defined

In [2]:
df_new = dataCleaner.filter_variables(dataset, list(zip_files.keys())[0], 'raw')

NameError: name 'dataCleaner' is not defined

In [5]:
# Important columns
columns_must_be_verified = ['APOE4', 'MMSE', 'Ventricles', 'Hippocampus', 'AGE']
single_column_required = ['DX']

In [6]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
processed_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
processed_df = dataCleaner.drop_if_all_none(processed_df, single_column_required)
processed_df.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
processed_df['VISCODE'] = processed_df['VISCODE'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [ ]:
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
processed_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DX')

## MMSE

In [9]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'MMSE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

dataset = zip_files[list(zip_files.keys())[0]]

In [10]:
df_new = dataCleaner.filter_variables(dataset, list(zip_files.keys())[0], 'raw')

In [11]:
# Important columns
columns_must_be_verified = ['MMSCORE']

In [12]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'VISDATE')
processed_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
processed_df['VISCODE2'] = processed_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [ ]:
filtered_df = processed_df[(processed_df['VISCODE2'] == 'sc') | (processed_df['VISCODE2'] == 'f')]
final_df = dataCleaner.handle_f_sc_values(filtered_df, processed_df, 'VISCODE2')

In [ ]:
final_df.drop(columns=['VISCODE'], inplace=True)

## PTDEMOG

In [ ]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'PTDEMOG'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

dataset = zip_files[list(zip_files.keys())[0]]

In [ ]:
df_new = dataCleaner.filter_variables(dataset, list(zip_files.keys())[0], 'raw')

In [ ]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTADDX']

In [ ]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'VISDATE')
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [ ]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [ ]:
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOB'], birth_year=row['PTDOBYY']), axis=1)

In [ ]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
final_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')

## ADSP_PHC_BIOMARKER

In [2]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'ADSP_PHC_BIOMARKER'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

dataset = zip_files[list(zip_files.keys())[0]]

In [3]:
dataset.columns

Index(['RID', 'PTID', 'SUBJID', 'PHASE', 'VISCODE', 'VISCODE2', 'DRAWDATE',
       'PHC_Visit', 'PHC_Age_Biomarker', 'PHC_Age_Cognition', 'PHC_Diagnosis',
       'PHC_Sex', 'PHC_Race', 'PHC_Ethnicity', 'PHC_Education', 'AB42_RAW',
       'PHC_AB42', 'Tau_RAW', 'PHC_Tau', 'pTau_RAW', 'PHC_pTau', 'AT_class',
       'Platform', 'PHC_SCeNS_AB42_Score', 'PHC_SCeNS_pTau_Score',
       'update_stamp'],
      dtype='object')

In [4]:
df_new = dataCleaner.filter_variables(dataset, list(zip_files.keys())[0], 'raw')

In [ ]:
columns_must_be_verified = ['PHC_Tau', 'PHC_pTau', 'PHC_AB42', 'AT_class']
single_column_required = ['PHC_Diagnosis']

In [5]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [7]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [ ]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PHC_Sex')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PHC_Education')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PHC_Ethnicity')
processed_df = dataCleaner.convert_to_two_bit(processed_df, col_name='AT_class')
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='PHC_Diagnosis')

In [ ]:
# non funziona la funzione perche black e Native Hawaian or PI sono invertite
mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 4: 3, 4.0: 3, 3: 4, 3.0: 4, 5: 5, 5.0: 5}
final_df['PHC_Race'] = final_df['PHC_Race'].map(mapping)

In [ ]:
final_df.drop(columns=['VISCODE'], inplace=True)

## BLCHANGE

In [2]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'BLCHANGE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

dataset = zip_files[list(zip_files.keys())[0]]

In [3]:
df_new = dataCleaner.filter_variables(dataset, list(zip_files.keys())[0], 'raw')

In [4]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [5]:
columns_must_be_verified = ['BCMMSE', 'BCADAS', 'BCPREDX']

In [ ]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'EXAMDATE')
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [9]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [ ]:
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='BCPREDX')

In [ ]:
final_df.drop(columns=['VISCODE'], inplace=True)

## DXSUM

In [4]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'DXSUM'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

dataset = zip_files[list(zip_files.keys())[0]]

In [5]:
df_new = dataCleaner.filter_variables(dataset, list(zip_files.keys())[0], 'raw')

In [12]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [13]:
columns_must_be_verified = ['DXNORM', 'DXMCI', 'DXNODEP']
required_column = ['DIAGNOSIS']

In [14]:
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'EXAMDATE')
no_none_df = dataCleaner.drop_if_all_none(no_none_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, required_column)
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: dataCleaner.convert_visitcode_to_int(x))

In [15]:
filtered_df = no_none_df[(no_none_df['VISCODE2'] == 'sc') | (no_none_df['VISCODE2'] == 'f')]
processed_df = dataCleaner.handle_f_sc_values(filtered_df, no_none_df, 'VISCODE2')

In [16]:
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DIAGNOSIS')

In [17]:
final_df.drop(columns=['VISCODE'], inplace=True)